# 정상 / 언더필 분류 — Colab 학습 (KEEP 3,425장 · 마스킹 · 500px)

**준비:** Drive `산공종설/` 폴더에 `keep3425_masked_500.zip`을 올린다. `train_pilot_classifier.py`(ConvNeXt·`--fast` 추가본)도 **Drive의 `산공종설/` 폴더**에 올린다. zip 안에 든 옛 버전 대신 쓰인다.
Colab 파일 패널 맨 위(`/content`)에 올리면 Drive가 아니다. 패널에서 `drive > MyDrive > 산공종설`을 연 뒤 그 안에 끌어다 놓는다.
그다음 위에서부터 셀을 하나씩 실행한다(Shift+Enter).

zip 안에는 세 가지가 있다.
- 마스킹 크롭 3,425장: normal 1714장, underfill_50 1711장
- `folds.csv`: 시간 블록 5-fold 분할표
- `train_pilot_classifier.py`: 학습 스크립트

⚠️ 여기서 나오는 점수는 **예비 점수**다. 클래스마다 연속 촬영 한 번 안에서 시간 구간을 나눠 검증한 것이라, 새 출력 실험에서의 성능이 아니다.

## 1. GPU 확인
런타임이 GPU인지 확인한다. `assert`는 조건이 거짓이면 그 자리에서 멈추고 메시지를 띄운다. 모르고 CPU로 몇 시간 돌리는 걸 막아 준다.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch, torchvision
print("torch", torch.__version__, "| torchvision", torchvision.__version__, "| cuda", torch.cuda.is_available())
assert torch.cuda.is_available(), "GPU 런타임이 아님: 런타임 > 런타임 유형 변경 > GPU"

## 2. Drive 연결
Colab은 실행할 때마다 빈 컴퓨터를 새로 빌려 준다. 그래서 Drive를 붙여야 내 파일이 보인다. 권한 창이 뜨면 허용한다.

결과는 Drive(`results/`)에 바로 저장한다. 런타임이 끊기면 `/content`에 있던 파일은 전부 사라지기 때문이다.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
DRIVE_DIR = "/content/drive/MyDrive/산공종설"   # zip을 올린 Drive 폴더
ZIP_NAME = "keep3425_masked_500.zip"
RESULTS_DIR = f"{DRIVE_DIR}/results"
print(os.listdir(DRIVE_DIR))
assert os.path.exists(f"{DRIVE_DIR}/{ZIP_NAME}"), "zip이 안 보임: DRIVE_DIR 경로를 확인"

## 3. zip 복사 → 압축 해제
Drive에서 사진 3천여 개를 하나씩 읽으면 느리다. 그래서 zip 하나를 Colab 로컬 디스크(`/content`)로 복사한 뒤 푼다.

- `!`로 시작하는 줄은 파이썬이 아니라 리눅스 셸 명령이다.
- `{DATA}`처럼 중괄호로 감싸면 파이썬 변수 값이 그 자리에 들어간다.
- 마지막 줄에 3425가 나와야 정상이다.
- Drive에 `train_pilot_classifier.py`가 있으면 zip 안 스크립트를 그걸로 바꾼다. 코드를 고칠 때마다 96MB zip을 다시 올리지 않아도 된다.

In [ ]:
DATA = "/content/data"
!mkdir -p {DATA}
!cp "{DRIVE_DIR}/{ZIP_NAME}" /content/
!unzip -q -o /content/{ZIP_NAME} -d {DATA}
!test -f "{DRIVE_DIR}/train_pilot_classifier.py" && cp "{DRIVE_DIR}/train_pilot_classifier.py" {DATA}/ && echo "Drive 스크립트 사용" || echo "zip 안 스크립트 사용 (ConvNeXt 없음)"
!grep -q -- "--fast" {DATA}/train_pilot_classifier.py && echo "스크립트 최신 (--fast 있음)" || echo "⚠ 옛 스크립트임: Drive 산공종설/ 의 train_pilot_classifier.py를 새 파일로 바꿀 것"
!ls {DATA}
!find {DATA}/size_500/masked_crops -name "*.jpg" | wc -l

## 4. 세 모델 1 epoch 점검 + 소요 시간 추정
`ARCHS`에 적은 모델마다 fold 0을 1 epoch씩 돌린다. 에러가 없는지 보고, 1 epoch 시간으로 전체 시간을 추정한다. 5·6번 셀도 이 `ARCHS` 목록을 그대로 쓴다.

| 모델 | 파라미터 | 특징 |
| --- | --- | --- |
| `efficientnet_b0` | 530만 | 가볍고 효율적 |
| `resnet18` | 1,170만 | 가장 단순한 고전 구조 |
| `convnext_tiny` | 2,800만 | 2020년대 설계. 연산량이 EffB0의 약 10배라 느림 |

- 여기 나오는 정확도는 1 epoch짜리라 의미 없다. **시간만 본다.**
**`FAST`:** `"--fast"`면 GPU가 입력 크기에 맞는 가장 빠른 계산 방법을 직접 골라 쓴다(`cudnn.benchmark`). 대신 같은 시드로 다시 돌려도 점수가 소수점 단위로 조금 달라질 수 있다. `""`(빈 문자열)이면 예전처럼 매번 똑같이 나오는 대신 느리다. 결과 폴더 이름에 `_fast`가 붙어서 두 방식의 결과가 섞이지 않는다.

- 전체 시간이 너무 길면 5번 셀 전에 `ARCHS`에서 모델을 빼거나 `SEEDS`를 줄인다.

In [ ]:
import json
ARCHS = ["efficientnet_b0", "resnet18", "convnext_tiny"]
FAST = "--fast"                      # 느려도 완전 재현이 필요하면 ""
TAG = "_fast" if FAST else ""
hours_per_seed = {}
for ARCH in ARCHS:
    !rm -rf /content/smoke_{ARCH}
    !python {DATA}/train_pilot_classifier.py --folds-csv {DATA}/folds.csv --pilot-root {DATA} --output /content/smoke_{ARCH} --arch {ARCH} --img-size 500 --folds 0 --seeds 0 --epochs 1 {FAST}
    sec = {m: json.load(open(f"/content/smoke_{ARCH}/runs/{m}/seed0/fold0/metrics.json"))["seconds"] for m in ("linear", "finetune")}
    # 1 epoch 시간 × 본 실행 epoch 수(linear 30, finetune 20) × 5 fold
    hours_per_seed[ARCH] = (sec["linear"] * 30 + sec["finetune"] * 20) * 5 / 3600
    print(f"=== {ARCH}: 1 epoch {sec} -> 시드 1개당 약 {hours_per_seed[ARCH]:.1f}시간")

print("\n시드 1개당 합계 약 %.1f시간, 시드 3개면 약 %.1f시간" % (sum(hours_per_seed.values()), 3 * sum(hours_per_seed.values())))

## 5. 본 실행 (모델 × 시드 자동 반복)
**무엇을 하나**
- **분할(folds.csv):** 클래스마다 사진을 프레임 번호순으로 줄 세워 5덩어리로 자른다. fold k는 k번째 덩어리로 검증하고 나머지로 학습한다. 검증 사진과 30프레임 이내인 학습 사진은 빼 둔다(buffer). 1초 간격 사진은 거의 똑같아서, 이웃 사진이 학습과 검증에 나뉘어 들어가면 점수가 부풀려지기 때문이다.
- **모드 2개:**
  - `linear`: ImageNet으로 미리 학습된 몸통은 고정하고, 마지막 분류층만 새로 학습한다. 빠르고 과적합이 적다.
  - `finetune`: 몸통까지 작은 학습률로 조금씩 고친다.
- **`--img-size 500`:** 500 크롭을 줄이지 않고 그대로 넣는다. 가는 틈이 뭉개지지 않는다.
- **시드:** 무작위 요소(초기값, 섞는 순서)만 바꿔 반복한다. 점수가 얼마나 흔들리는지 보는 용도다.

**반복 순서는 시드 먼저다.** 시드 0으로 세 모델을 다 돌린 뒤 시드 1로 넘어간다. 도중에 끊겨도 세 모델을 한 번씩은 비교할 수 있다.

**끊겼을 때**
- 다 끝난 조합은 `summary.csv`가 있어서 건너뛴다. 이 셀을 다시 실행하면 남은 것부터 한다.
- 돌다가 끊긴 조합의 폴더는 결과가 반만 있어서 스크립트가 "output folder is not empty"로 멈춘다. 그 폴더만 Drive에서 지우고 다시 실행한다.

In [ ]:
import os
SEEDS = ["0", "1", "2"]
for SEED in SEEDS:
    for ARCH in ARCHS:
        OUT = f"{RESULTS_DIR}/{ARCH}_keep3425_masked_500{TAG}_seeds{SEED}"
        if os.path.exists(f"{OUT}/summary.csv"):
            print("이미 끝남:", OUT)
            continue
        print(f"\n===== {ARCH} seed {SEED} -> {OUT}")
        !python {DATA}/train_pilot_classifier.py --folds-csv {DATA}/folds.csv --pilot-root {DATA} --output "{OUT}" --arch {ARCH} --img-size 500 --seeds {SEED} {FAST}

## 6. 결과 모아 보기
끝난 조합만 모아 표로 만든다. 도중에 실행해도 된다.

**표 읽는 법**
- 첫 번째 표는 모델·시드·모드별 점수다. 두 번째 표는 시드를 평균한 점수다.
- `pooled_…`: 5개 fold의 검증 예측을 모아서 한 번에 계산한 점수다. 모든 사진이 한 번씩 검증된다.
- `underfill_recall`: 실제 언더필 사진 중 모델이 잡아낸 비율이다. 불량을 놓치지 않는 정도다.
- `normal_recall`: 실제 정상 사진을 정상으로 맞힌 비율이다.
- `auroc`: 판정 기준(임계값)과 상관없이 두 클래스를 얼마나 잘 가르는지 나타낸다. 0.5는 찍기 수준, 1은 완벽이다.
- `fold_std_accuracy`: fold 사이의 정확도 흔들림이다. 크면 시간 구간에 따라 성능이 들쭉날쭉하다는 뜻이다.
- `train_accuracy`는 1.0 가까운데 검증 점수가 훨씬 낮으면 과적합(학습 사진을 외운 것)이다.

**shortcut_check**
마스킹 비율이나 평균 밝기 하나만으로 클래스가 갈리는지 본다. 모델과 상관없는 데이터 점검이라 한 번만 출력한다.
- AUROC가 0.5 근처면 괜찮다.
- 0이나 1에 가까우면, 모델이 결함 대신 그 차이를 보고 맞힐 위험이 있다.

fold별 상세, 오분류 목록, 가중치는 Drive의 각 결과 폴더에 있다. **이 셀 출력을 그대로 복사해서 Claude에게 붙여 주면 된다.**

In [ ]:
import pandas as pd
cols = ["arch", "seed", "mode", "pooled_mean_accuracy", "pooled_mean_underfill_recall", "pooled_mean_underfill_f1",
        "pooled_mean_normal_recall", "pooled_mean_auroc", "fold_std_accuracy", "mean_train_accuracy",
        "mean_seconds_per_run", "max_peak_alloc_mib"]
parts, done = [], []
for ARCH in ARCHS:
    for SEED in SEEDS:
        OUT = f"{RESULTS_DIR}/{ARCH}_keep3425_masked_500{TAG}_seeds{SEED}"
        if os.path.exists(f"{OUT}/summary.csv"):
            s = pd.read_csv(f"{OUT}/summary.csv", encoding="utf-8-sig")
            s.insert(0, "arch", ARCH)
            s.insert(1, "seed", SEED)
            parts.append(s)
            done.append(OUT)
if not parts:
    print("아직 끝난 조합이 없음")
else:
    tab = pd.concat(parts)[cols]
    print(tab.round(3).to_string(index=False))
    print("\n[시드 평균]")
    g = tab.drop(columns="seed").groupby(["arch", "mode"])
    avg = g.mean().round(3)
    avg.insert(0, "n_seeds", g.size())   # 몇 개 시드의 평균인지
    print(avg.to_string())
    print("\n[shortcut_check]\n" + open(f"{done[0]}/shortcut_check.json").read())